# Sprint 1: Automated Data Pipeline & Forensic Log Analyzer

### Project Mission
You are working as an **Analytics Engineer / Data Analyst**. Microservices generate daily server logs in `data/daily_logs/`. These files arrive in a mix of `.txt` and `.json` formats, and several files are corrupted.

If an automated data pipeline crashes on corrupted data, morning business dashboards fail. Your goal is to build an automated, crash-proof pipeline that filters junk files, safely parses data with Object-Oriented Programming (OOP), aggregates forensic server errors (`5xx`), consolidates clean data into `analytics_ready.json`, and quarantines corrupted files into `data/quarantine/`.

---

### Pipeline Architecture Flow

```text
+-----------------------------------------------------------------------------+
|                              data/daily_logs/                               |
|              (75 mixed .txt & .json log files + hidden/junk files)          |
+--------------------------------------┬--------------------------------------+
                                       |
                                       v  Step 1: is_valid_logfile()
+-----------------------------------------------------------------------------+
|                         Candidate Log Files (72 files)                      |
|                    (Skips hidden '.', backup '~', and .md)                  |
+--------------------------------------┬--------------------------------------+
                                       |
                                       v  Step 2: DataParser Validation
                          +-------------------------+
                          |  Try-Except Validation  |
                          +------------┬------------+
                                       |
                  +--------------------+--------------------+
                  v (Valid Records)                         v (Corrupted Files)
     +-------------------------+               +-------------------------+
     |      clean_records      |               |     quarantine_list     |
     |      (65 records)       |               |        (7 files)        |
     +------------┬------------+               +------------┬------------+
                  |                                         |
   Step 3: 5xx?   | Counter()                  Step 4: copy |
                  v                                         v
     +-------------------------+               +-------------------------+
     |    error_counts (IPs)   |               |    data/quarantine/     |
     |  (Top Offending IPs)    |               |  (Preserved for Audit)  |
     +-------------------------+               +-------------------------+
                  |
     json.dump()  |
                  v
     +-------------------------+
     |  analytics_ready.json   |
     +-------------------------+
```


## Q1: How do we scan a directory and filter out hidden/junk files?

### Concept Pointers:
- Operating systems create hidden files (such as `.DS_Store`), and editors create temporary backup files (such as `~backup.json`).
- `os.listdir()` returns all filenames in a folder.
- We filter out files starting with `.` or `~` and only accept `.txt` or `.json`.

In [1]:
import os

LOG_DIR = "data/daily_logs"

def is_valid_logfile(filename):
    """Check if filename is a valid .txt or .json log, ignoring hidden/junk files."""
    if filename.startswith(".") or filename.startswith("~"):
        return False
    return filename.lower().endswith((".txt", ".json"))


In [2]:
# Scan directory using our filter
all_entries = os.listdir(LOG_DIR)
data_files = [os.path.join(LOG_DIR, f) for f in all_entries if is_valid_logfile(f)]
data_files.sort()

print(f"Total items in folder:     {len(all_entries)}")
print(f"Candidate log files:       {len(data_files)}")
print(f"Junk/hidden items skipped: {len(all_entries) - len(data_files)}")


Total items in folder:     74
Candidate log files:       72
Junk/hidden items skipped: 2


### Key Takeaways:
- Out of 75 files in `data/daily_logs/`, exactly 72 are valid candidate logs.
- 3 junk files (`.DS_Store`, `~backup_event_014.json`, `notes.md`) were filtered before attempting any parsing.

## Q2: How do we safely parse a single JSON log file?

### Concept Pointers:
- JSON files can fail from **malformed syntax** (`json.JSONDecodeError`) or **missing required keys** (`KeyError`).
- We catch both in a targeted `try-except` block.

In [3]:
import json

def parse_json_file(filepath):
    """Safely parse a single JSON log file."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        record = {"source": filepath, "ip": data["ip_address"], "status": data["status_code"]}
        return record, None
    except (json.JSONDecodeError, KeyError) as e:
        return None, f"{type(e).__name__}: {e}"


In [4]:
# Test on a known JSON file
sample_json = [f for f in data_files if f.endswith(".json")][0]
record, error = parse_json_file(sample_json)
print(f"File:   {os.path.basename(sample_json)}")
print(f"Record: {record}")
print(f"Error:  {error}")


File:   event_000.json
Record: {'source': 'data/daily_logs\\event_000.json', 'ip': '203.0.113.55', 'status': 200}
Error:  None


## Q3: How do we safely parse a single TXT log file?

### Concept Pointers:
- TXT logs are formatted as comma-separated values: `IP,STATUS`.
- If a line is missing the comma delimiter, `line.split(',')` raises a `ValueError`.

In [5]:
def parse_txt_file(filepath):
    """Safely parse a single comma-delimited TXT log file."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            line = f.readline().strip()
        ip, status = line.split(",")
        record = {"source": filepath, "ip": ip.strip(), "status": status.strip()}
        return record, None
    except ValueError as e:
        return None, f"{type(e).__name__}: {e}"


In [6]:
# Test on a known TXT file
sample_txt = [f for f in data_files if f.endswith(".txt")][0]
record, error = parse_txt_file(sample_txt)
print(f"File:   {os.path.basename(sample_txt)}")
print(f"Record: {record}")
print(f"Error:  {error}")


File:   access_040.txt
Record: {'source': 'data/daily_logs\\access_040.txt', 'ip': '192.168.1.14', 'status': '503'}
Error:  None


## Q4: How do we encapsulate this into a clean `DataParser` class (OOP)?

### Concept Pointers:
- **Single Responsibility Principle**: The class manages validation state (`is_valid`, `record`, `error`) for an individual file.

In [7]:
class DataParser:
    """Object-Oriented parser for an individual log file."""
    def __init__(self, filepath):
        self.filepath = filepath
        self.is_valid = True
        self.record = None
        self.error = None

    def parse(self):
        if self.filepath.lower().endswith(".json"):
            self.record, self.error = parse_json_file(self.filepath)
        else:
            self.record, self.error = parse_txt_file(self.filepath)
            
        if self.error is not None:
            self.is_valid = False
        return self.record


In [8]:
# Test the class on all candidate files and report corrupted files
print("--- Corrupted Files Audit ---")
bad_count = 0
for path in data_files:
    parser = DataParser(path)
    parser.parse()
    if not parser.is_valid:
        bad_count += 1
        print(f"[FLAGGED] {os.path.basename(path):26s} -> Reason: {parser.error}")

print(f"\nTotal corrupted files identified: {bad_count}")


--- Corrupted Files Audit ---
[FLAGGED] access_070.txt             -> Reason: ValueError: not enough values to unpack (expected 2, got 1)
[FLAGGED] access_071.txt             -> Reason: ValueError: not enough values to unpack (expected 2, got 1)
[FLAGGED] event_065.json             -> Reason: JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 87 (char 86)
[FLAGGED] event_066.json             -> Reason: JSONDecodeError: Expecting ',' delimiter: line 1 column 37 (char 36)
[FLAGGED] event_067.json             -> Reason: KeyError: 'status_code'
[FLAGGED] event_068.json             -> Reason: KeyError: 'status_code'
[FLAGGED] event_069.json             -> Reason: KeyError: 'ip_address'

Total corrupted files identified: 7


## Q5: How do we process all files and aggregate 5xx forensic anomalies?

### Concept Pointers:
- `5xx` status codes indicate server errors (such as 500 Internal Server Error, 502 Bad Gateway).
- We use Python's `collections.Counter` to track anomaly counts per IP address.

In [9]:
from collections import Counter

clean_records = []
quarantine_list = []
error_counts = Counter()

for path in data_files:
    parser = DataParser(path)
    parser.parse()
    
    if parser.is_valid:
        clean_records.append(parser.record)
        # Tally 5xx errors
        if str(parser.record["status"]).startswith("5"):
            error_counts[parser.record["ip"]] += 1
    else:
        quarantine_list.append(path)


In [10]:
print(f"Clean records collected:      {len(clean_records)}")
print(f"Corrupt files for quarantine: {len(quarantine_list)}\n")
print("Top 5 Offending IPs (5xx Server Errors):")
for ip, count in error_counts.most_common(5):
    print(f"  IP: {ip:16s} | Errors: {count}")


Clean records collected:      65
Corrupt files for quarantine: 7

Top 5 Offending IPs (5xx Server Errors):
  IP: 45.33.32.156     | Errors: 3
  IP: 198.51.100.23    | Errors: 3
  IP: 10.0.0.8         | Errors: 3
  IP: 192.168.1.14     | Errors: 2
  IP: 203.0.113.55     | Errors: 2


## Q6: How do we save the clean data and isolate the corrupt files?

### Concept Pointers:
- Valid records are written to `analytics_ready.json` via `json.dump()`.
- Corrupted files are copied to `data/quarantine/` using `shutil.copy()` for audit retention.

In [11]:
import shutil

# 1. Save consolidated clean records
OUTPUT_JSON = "analytics_ready.json"
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(clean_records, f, indent=2)
print(f"[OUTPUT] Saved {len(clean_records)} clean records to '{OUTPUT_JSON}'")


[OUTPUT] Saved 65 clean records to 'analytics_ready.json'


In [12]:
# 2. Isolate corrupted files into quarantine directory
QUARANTINE_DIR = "data/quarantine"
os.makedirs(QUARANTINE_DIR, exist_ok=True)

for bad_file in quarantine_list:
    dest = os.path.join(QUARANTINE_DIR, os.path.basename(bad_file))
    shutil.copy(bad_file, dest)

print(f"[QUARANTINE] Isolated {len(quarantine_list)} corrupted files in '{QUARANTINE_DIR}/'")


[QUARANTINE] Isolated 7 corrupted files in 'data/quarantine/'


## Q7: How do we programmatically verify data integrity on disk?

### Concept Pointers:
- Use `assert` statements to ensure disk artifacts match memory state.

In [13]:
# Programmatic assertion checks
with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
    saved_data = json.load(f)
assert len(saved_data) == len(clean_records), "Record count mismatch!"
assert len(os.listdir(QUARANTINE_DIR)) == len(quarantine_list), "Quarantine count mismatch!"

print("[VERIFIED] All programmatic verification assertions passed!")
print(f"Sample clean record: {saved_data[0]}")


[VERIFIED] All programmatic verification assertions passed!
Sample clean record: {'source': 'data/daily_logs\\access_040.txt', 'ip': '192.168.1.14', 'status': '503'}


## Summary & Key Takeaways

- **75 total files**: 3 non-data junk files safely ignored, 72 candidate logs parsed.
- **65 clean records**: Formatted and exported to `analytics_ready.json`.
- **7 corrupted files**: Intercepted without crashing and quarantined in `data/quarantine/`.
- **Next Sprint**: We transition into live Web Scraping, NumPy data cleaning, and Exploratory Data Analysis!